In [ ]:
'''
python version 3.10.12
'''

In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import json  # used to read config file
import AASIST  # official AASIST implementation from https://github.com/clovaai/aasist/blob/main/models/AASIST.py
from tqdm import tqdm  # progress bar
from Model import  DownStreamLinearClassifier # SSDNet is the Res-TSSDNet Model
from DatasetUtils import genSpoof_list, Dataset_ASVspoof2019_train  # ASVspoof dataset utils
# import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
batch_size = 24
gpu = 0  # GPU id to use
torch.cuda.set_device(gpu)

# Load Config file
with open("./config.conf", "r") as f_json:
    config = json.loads(f_json.read())

In [3]:
def load_model(model_name:str, config:dict):
    
    with open(config['aasist_config_path'], "r") as f_json:        
        aasist_config = json.loads(f_json.read())
    aasist_model_config = aasist_config["model_config"]
    aasist_encoder = AASIST.AasistEncoder(aasist_model_config).to(device)
    downstream_model = DownStreamLinearClassifier(aasist_encoder, input_depth=160)
    checkpoint = torch.load(config['clad_model_path_for_evaluation'], map_location=device)
    downstream_model.load_state_dict(checkpoint["state_dict"])
    downstream_model = downstream_model.to(device)
    return downstream_model

In [4]:
def evaluation_19_LA_eval(eval_num,model, score_save_path, model_name, database_path,eval_path, augmentations=None, augmentations_on_cpu=None, batch_size = 24, manipulation_on_real=True, cut_length = 64600):
    # In asvspoof dataset, label = 1 means bonafide.
    model.eval()
    device = "cuda"
  
    variants, file_eval,randoms,src,label_list  = genSpoof_list(eval_path, is_train=False, is_eval=False)
    print('no. of ASVspoof 2019 LA evaluating trials', len(file_eval))
   
    asvspoof_LA_eval_dataset = Dataset_ASVspoof2019_train(list_IDs=file_eval, label_IDs=label_list,random_IDs=randoms,src_IDs=src,variant_IDs=variants, base_dir=database_path, cut_length=cut_length)
    asvspoof_2019_LA_eval_dataloader = DataLoader(asvspoof_LA_eval_dataset, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=8, pin_memory=True)  # added num_workders param to speed up.
    with open(score_save_path, 'w') as file:  # This creates an empty file or empties an existing file
        pass
    
    with torch.no_grad():
        score_list = []
        label_list=[]  
        audio_file_list=[]
        variant_list=[]
        src_list=[]
        random_list=[]
        # for batch_idx, (audio_input, spks, labels) in enumerate(tqdm(asvspoof_2019_LA_eval_dataloader)):
        for batch_idx, (audio_input, labels, audio_file,variant,src,random_label) in enumerate(tqdm(asvspoof_2019_LA_eval_dataloader)):
            
            
            audio_input = audio_input.squeeze(1)
            if augmentations_on_cpu != None:
                audio_input = augmentations_on_cpu(audio_input)
            
            audio_input = audio_input.to(device)

            
            if audio_input.shape[-1] < cut_length:
                audio_input = audio_input.repeat(1, int(cut_length/audio_input.shape[-1])+1)[:, :cut_length]
            elif audio_input.shape[-1] > cut_length:
                audio_input = audio_input[:, :cut_length]
            
            batch_out = model(audio_input)
            
            batch_score = (batch_out[:, 1]).data.cpu().numpy().ravel()
            label_l = ['bonafide' if i==1 else 'spoof' for i in labels]
            label_list.extend(label_l)
            score_list.extend(batch_score.tolist())
            audio_file_list.extend(audio_file)
            variant_list.extend(variant)
            src_list.extend(src)
            random_list.extend(random_label)
        
        with open(score_save_path, 'w') as fh:
            for v,pa,ra,sr,label, cm_score in zip(variant_list,audio_file_list,random_list,src_list,label_list,score_list):
                fh.write('{} {} {} {} {} {}\n'.format(v,pa,ra,sr,label, cm_score))
            fh.close()   
        print('Scores saved to {}'.format(score_save_path))

In [ ]:
cut_length = 64600
model_name='CLAD'
model = load_model(model_name,config)
'''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
'''
database_path='path to VoiceWukong dataset'
evaluation_19_LA_eval(2,model=model, 
                      model_name=model_name, 
                      database_pat=database_path,
                      eval_path='path to eval_list.txt', 
                      batch_size =24,  
                      score_save_path='path to save en_eval_score.txt', # /yourpath/en_eval_score.txt
                      cut_length=cut_length)
evaluation_19_LA_eval(2,model=model, model_name=model_name, 
                      database_path=database_path,
                      eval_path='path to zh_eval_list.txt', 
                      batch_size =24,  score_save_path='path to save zh_eval_score.txt', 
                      cut_length=cut_length)